
# Chapter 7: Transformations

Source orientation: printed pages 143-173, course-mapped PDF pages 153-183, sections 7.1-7.9. This notebook uses the source for topic order and coverage only. The prose, examples, code, diagrams, checks, and lab artifacts are original to this course notebook.

## Chapter Goal

The chapter's working question is: **if a geometry is understood by its transformations, what does each transformation group keep meaningful?** A Euclidean isometry protects distance. An affine transformation protects parallelism and ratios along a line, but not length or angle. A projective-line map protects cross-ratio, while allowing ordinary distances and parallelism to disappear. On the sphere, isometries become restrictions of origin-fixing isometries of space, and the orientation-preserving ones are rotations. Quaternions then turn those rotations into algebra, at the cost of identifying antipodal unit quaternions.

The notebook moves through the same ladder computationally:

| Chapter idea | Computational translation | Invariant to inspect |
| --- | --- | --- |
| Isometry group of the plane | Reflections, rotations, and translations as matrices plus shifts | Pairwise distances; orientation only in the even-reflection subgroup |
| Linear and affine maps | `u -> M u` and `u -> M u + c` | Vector operations for linear maps; parallelism, midpoints, and ratios on a line for affine maps |
| Projective line | Lines through the origin in `R^2`; matrices up to nonzero scalar | Cross-ratio of four labels on `RP^1` |
| Spherical geometry | Unit sphere, great circles, plane reflections through the origin | Great-circle distance; products of two reflections are rotations |
| Quaternion rotations | Unit quaternions acting by `p -> q p q^{-1}` on imaginary quaternions | Norms, axes, composition order, and the double cover `q ~ -q` |
| Finite rotation groups | Tetrahedral rotations represented by 24 unit quaternions | 12 antipodal pairs; vertices of the 24-cell on `S^3` |
| `S^3` and `RP^3` | Unit quaternions and antipodal pairs | `S^3` is a group; `RP^3` models `SO(3)` |

The inspection habit is the same in every section: identify the allowed transformations, watch what changes, and check the claimed invariant numerically or symbolically.


In [ ]:

from pathlib import Path
import sys, json, math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, FancyArrowPatch, Polygon
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp
import trimesh


def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "The-Four-Pillars-of-Geometry" and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Could not locate The-Four-Pillars-of-Geometry root")


BOOK_ROOT = find_book_root(Path.cwd().resolve())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, assert_artifacts, display_artifact, image_stats, save_json, save_table
from utils.transforms import q_from_axis_angle, q_multiply, q_rotate

CHAPTER_KEY = "chapter-07-transformations"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_KEY
for category in ["figures", "html", "checks", "tables"]:
    (ARTIFACT_ROOT / category).mkdir(parents=True, exist_ok=True)

# This notebook replaces the old generic scaffold with direct, chapter-specific cells.

PALETTE = {
    "ink": "#243040", "blue": "#1f77b4", "orange": "#e07a2f", "green": "#2e8b57",
    "red": "#c74343", "purple": "#6f5bd7", "gold": "#c7971d", "gray": "#78818c",
    "light": "#f6f2e8", "paper": "#fffdf7",
}
figure_paths, html_paths, check_paths, table_paths = [], [], [], []
chapter_checks = {}


def rel(path: Path) -> str:
    return path.relative_to(BOOK_ROOT).as_posix()


def save_figure(fig, filename: str) -> Path:
    path = artifact_path(ARTIFACT_ROOT, "figures", filename)
    fig.savefig(path, dpi=170, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    figure_paths.append(path)
    return path


def equal_2d(ax):
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.8)
    ax.set_facecolor(PALETTE["paper"])


def pairwise_distances(points):
    pts = np.asarray(points, dtype=float)
    return np.array([np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i + 1, len(pts))])


print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT}")



## Storyboard

The storyboard below is the implementation contract for the rest of the notebook. Every visual has a target for the reader and a check that keeps the picture honest.

| Order | Concept visual | Artifact | What to inspect | Check |
| --- | --- | --- | --- | --- |
| 1 | Group/invariant map | `group-invariants-lattice.png` | The smaller the group, the more structure may become meaningful | Route table has every major chapter group and invariant |
| 2 | Plane isometry composition | `plane-isometry-reflection-composition.png` | Two reflections preserve distance and become a rotation; order matters | Distance residual, determinants, noncommuting products |
| 3 | Affine/vector grid | `affine-vector-grid-invariants.png` | Parallelism, midpoints, and line ratios survive; lengths and angles need not | Determinant area scale, midpoint and ratio residuals |
| 4 | Projective-line map | `projective-line-mobius-cross-ratio.png` | A matrix moves lines through the origin; scaling the matrix changes no `RP^1` point | Exact SymPy cross-ratio equality and scalar-matrix check |
| 5 | Spherical isometries | `spherical-great-circle-rotations.png`, `spherical-rotation-explorer.html` | Great circles are plane sections; two plane reflections rotate around the intersection axis | Orthogonality, determinant, and great-circle distance residuals |
| 6 | Quaternion rotation cover | `quaternion-double-cover-axis-angle.png` | Half angles in `q`, full angle in `q p q^{-1}`, and `q` / `-q` give the same rotation | Unit norm, matrix-vs-quaternion residual, double-cover residual |
| 7 | Applied rotation lab | `applied-quaternion-rotation-lab.html` | Changing the order of two rotations changes the final frame | Orthonormal frame checks and order-difference norm |
| 8 | Tetrahedral rotations | `tetrahedral-rotation-quaternion-cloud.png` | The 12 rotations lift to 24 unit quaternions, the 24-cell vertex set | Count, norm, antipodal-pair, and tetrahedron mesh checks |
| 9 | `S^3` and `RP^3` | `s3-rp3-antipodal-model.png` | A 2-pi rotation reaches the antipode in `S^3` but the same point in `RP^3` | Quaternion multiplication stays on `S^3`; antipodal rotations agree |


## 1. Group View: Which Properties Are Meaningful?

Klein's viewpoint is easy to misread as a slogan unless the invariants are made explicit. The diagram in this section treats each geometry as a choice of transformation group. A property is part of that geometry when every allowed transformation preserves it.

The arrows are not a theorem hierarchy in all possible settings; they are a chapter map. For example, affine transformations include the linear maps after we stop treating the origin as special. Projective maps on `RP^1` are represented by ordinary `2 x 2` matrices only after we agree that a whole line through the origin is one projective point. The quaternion rows are different again: the group itself becomes a geometric object.


In [ ]:
group_rows = [
    {"group": "Isom(R^2)", "model": "distance-preserving maps", "keeps": "distance, lines, circles", "does_not_keep": "absolute vertical direction"},
    {"group": "Isom+(R^2)", "model": "even products of reflections", "keeps": "distance and orientation", "does_not_keep": "mirror handedness reversal"},
    {"group": "GL(2,R)", "model": "invertible linear maps", "keeps": "origin, vector operations, lines through origin", "does_not_keep": "most distances and angles"},
    {"group": "Aff(2)", "model": "u -> M u + c", "keeps": "parallelism, midpoints, ratios on a line", "does_not_keep": "origin, length, angle"},
    {"group": "PGL(2,R) on RP^1", "model": "linear fractional maps", "keeps": "cross-ratio", "does_not_keep": "affine distance and parallelism"},
    {"group": "Isom(S^2)", "model": "great-circle distance isometries", "keeps": "great circles and spherical distance", "does_not_keep": "orientation if reflections allowed"},
    {"group": "SO(3)", "model": "orientation-preserving sphere isometries", "keeps": "axis-angle rotations and orientation", "does_not_keep": "commutativity"},
    {"group": "S^3 unit quaternions", "model": "unit quaternions under product", "keeps": "quaternion norm and group product", "does_not_keep": "one-to-one rotation labels"},
    {"group": "RP^3", "model": "antipodal pairs of unit quaternions", "keeps": "space rotations", "does_not_keep": "choice between q and -q"},
]

table_path = save_table(group_rows, ARTIFACT_ROOT, "tables", "group_invariant_route.csv")
table_paths.append(table_path)
G = nx.DiGraph()
for row in group_rows:
    G.add_node(row["group"], keeps=row["keeps"])
G.add_edges_from([
    ("Isom(R^2)", "Isom+(R^2)"), ("GL(2,R)", "Aff(2)"),
    ("GL(2,R)", "PGL(2,R) on RP^1"), ("Isom(S^2)", "SO(3)"),
    ("SO(3)", "RP^3"), ("S^3 unit quaternions", "RP^3"),
])
pos = {
    "Isom(R^2)": (-2.6, 1.6), "Isom+(R^2)": (-2.6, 0.35),
    "GL(2,R)": (-0.6, 1.6), "Aff(2)": (-0.6, 0.35),
    "PGL(2,R) on RP^1": (-0.6, -0.9), "Isom(S^2)": (1.5, 1.6),
    "SO(3)": (1.5, 0.35), "S^3 unit quaternions": (3.5, 1.6), "RP^3": (3.5, 0.35),
}
node_color = {name: color for names, color in [
    (["Isom(R^2)", "Isom+(R^2)"], PALETTE["blue"]),
    (["GL(2,R)", "Aff(2)"], PALETTE["orange"]),
    (["PGL(2,R) on RP^1"], PALETTE["purple"]),
    (["Isom(S^2)", "SO(3)"], PALETTE["green"]),
    (["S^3 unit quaternions", "RP^3"], PALETTE["red"]),
] for name in names}
fig, ax = plt.subplots(figsize=(13, 7), facecolor=PALETTE["paper"])
ax.set_facecolor(PALETTE["paper"])
for edge in G.edges:
    arrow = FancyArrowPatch(np.array(pos[edge[0]]), np.array(pos[edge[1]]), arrowstyle="-|>", mutation_scale=15, lw=1.8, color=PALETTE["gray"], alpha=0.85)
    ax.add_patch(arrow)
for node, (x, y) in pos.items():
    row = next(item for item in group_rows if item["group"] == node)
    text = f"{node}\nkeeps: {row['keeps']}\nnot fixed: {row['does_not_keep']}"
    ax.text(x, y, text, ha="center", va="center", fontsize=9.5, color=PALETTE["ink"], bbox={"boxstyle": "round,pad=0.45", "fc": "white", "ec": node_color[node], "lw": 2.0})
ax.text(-3.35, 2.25, "Transformation groups as invariant filters", fontsize=16, weight="bold", color=PALETTE["ink"])
ax.text(-3.35, -1.65, "Reading target: a property belongs to the geometry only if every allowed transformation preserves it.", fontsize=10.5, color=PALETTE["ink"])
ax.set_xlim(-3.55, 4.25); ax.set_ylim(-1.9, 2.55); ax.axis("off")
group_fig = save_figure(fig, "group-invariants-lattice.png")
chapter_checks["group_invariant_map"] = {"groups_recorded": len(group_rows), "edges_recorded": G.number_of_edges(), "has_rp3_row": True}
display_artifact(group_fig, width=900)
print(rel(table_path))


## 2. Plane Isometries: Reflections Generate Motion

The source chapter starts the group discussion with plane isometries because they make the word *group* concrete. Closure is visible: composing distance-preserving maps gives another distance-preserving map. Inverses are visible too, especially for reflections, since reflecting twice in the same line undoes itself.

The figure below uses two reflecting lines through the origin. One reflection reverses orientation and has determinant `-1`. Two reflections preserve orientation and produce a rotation through twice the angle between the reflecting lines. The triangle is deliberately asymmetric so that orientation reversal and rotation cannot be mistaken for a harmless relabeling.


In [ ]:
def reflection_line_matrix(alpha: float) -> np.ndarray:
    return np.array([[math.cos(2 * alpha), math.sin(2 * alpha)], [math.sin(2 * alpha), -math.cos(2 * alpha)]], dtype=float)
alpha = math.radians(18); beta = math.radians(62)
R_alpha = reflection_line_matrix(alpha); R_beta = reflection_line_matrix(beta)
rotation_from_two_reflections = R_beta @ R_alpha; reverse_order = R_alpha @ R_beta
rotation_angle = math.degrees(math.atan2(rotation_from_two_reflections[1, 0], rotation_from_two_reflections[0, 0]))
triangle = np.array([[0.35, 0.15], [1.35, 0.38], [0.72, 1.25]])
reflected = (R_alpha @ triangle.T).T; rotated = (rotation_from_two_reflections @ triangle.T).T; reverse_rotated = (reverse_order @ triangle.T).T
fig, ax = plt.subplots(figsize=(8.4, 7.2), facecolor=PALETTE["paper"]); equal_2d(ax)
for angle, label, color in [(alpha, "L_alpha", PALETTE["blue"]), (beta, "L_beta", PALETTE["orange"] )]:
    direction = np.array([math.cos(angle), math.sin(angle)]); pts = np.vstack([-2.0 * direction, 2.0 * direction])
    ax.plot(pts[:, 0], pts[:, 1], color=color, lw=2.0, label=label); ax.text(*(1.85 * direction), label, color=color, fontsize=10, weight="bold")
for pts, color, label, alpha_fill in [(triangle, PALETTE["ink"], "start", 0.08), (reflected, PALETTE["blue"], "after one reflection", 0.12), (rotated, PALETTE["green"], "after two reflections", 0.16)]:
    ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor=color, alpha=alpha_fill, lw=2.5))
    closed = np.vstack([pts, pts[0]]); ax.plot(closed[:, 0], closed[:, 1], color=color, lw=2.2, label=label)
    center = pts.mean(axis=0); ax.text(center[0], center[1], label, ha="center", va="center", fontsize=9, color=PALETTE["ink"], bbox={"fc": "white", "ec": "none", "alpha": 0.72, "pad": 2})
ax.add_patch(Arc((0, 0), 1.0, 1.0, theta1=math.degrees(alpha), theta2=math.degrees(beta), color=PALETTE["red"], lw=2))
ax.text(0.47, 0.45, "angle between mirrors", color=PALETTE["red"], fontsize=9)
ax.text(-1.85, -1.55, f"product angle = {rotation_angle:.1f} degrees = 2({math.degrees(beta-alpha):.1f})", color=PALETTE["ink"], fontsize=11)
ax.text(-1.85, -1.75, "reverse order rotates the other way; products of isometries need not commute", color=PALETTE["ink"], fontsize=10)
ax.set_xlim(-2.05, 2.2); ax.set_ylim(-1.95, 2.05)
ax.set_title("Two Reflections Preserve Distances But Encode Order", fontsize=15, weight="bold", color=PALETTE["ink"])
ax.legend(loc="upper left", frameon=True, fontsize=9)
isom_fig = save_figure(fig, "plane-isometry-reflection-composition.png")
orig_d = pairwise_distances(triangle); rot_d = pairwise_distances(rotated)
chapter_checks["plane_isometry_composition"] = {"distance_max_error": float(np.max(np.abs(orig_d - rot_d))), "det_reflection_alpha": float(np.linalg.det(R_alpha)), "det_reflection_beta": float(np.linalg.det(R_beta)), "det_two_reflections": float(np.linalg.det(rotation_from_two_reflections)), "rotation_angle_degrees": float(rotation_angle), "noncommuting_order_gap": float(np.linalg.norm(rotated - reverse_rotated))}
display_artifact(isom_fig, width=780)
chapter_checks["plane_isometry_composition"]


## 3. Vector and Affine Transformations

A linear transformation is the computational form of preserving vector addition and scalar multiplication. That is stronger than preserving straightness: it also keeps the origin fixed. The affine step `u -> M u + c` removes the artificial privilege of the origin while retaining the linear behavior inside differences of points.

The grid view is useful because affine invariants are easiest to see in families. Parallel lines stay parallel. Midpoints and ratios on one line stay put in the correct affine sense. The unit circle becomes an ellipse, so the same visual also warns against treating affine geometry as Euclidean geometry.


In [ ]:
M_aff = np.array([[1.35, 0.58], [-0.28, 0.92]], dtype=float); c_aff = np.array([0.72, 0.18], dtype=float)
def affine_map(points):
    pts = np.asarray(points, dtype=float); return (M_aff @ pts.T).T + c_aff
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.8), facecolor=PALETTE["paper"])
for ax in axes: equal_2d(ax)
ts = np.linspace(-1.0, 1.0, 7); line_t = np.linspace(-1.0, 1.0, 80)
for value in ts:
    axes[0].plot(line_t, np.full_like(line_t, value), color=PALETTE["gray"], lw=0.8); axes[0].plot(np.full_like(line_t, value), line_t, color=PALETTE["gray"], lw=0.8)
angles = np.linspace(0, 2 * math.pi, 240); circle = np.c_[np.cos(angles), np.sin(angles)]
axes[0].plot(circle[:, 0], circle[:, 1], color=PALETTE["blue"], lw=2.2, label="unit circle"); axes[0].set_title("Before: vector grid", color=PALETTE["ink"], weight="bold"); axes[0].set_xlim(-1.25, 1.25); axes[0].set_ylim(-1.25, 1.25)
for value in ts:
    for line in [np.c_[line_t, np.full_like(line_t, value)], np.c_[np.full_like(line_t, value), line_t]]:
        img = affine_map(line); axes[1].plot(img[:, 0], img[:, 1], color=PALETTE["gray"], lw=0.8)
ellipse = affine_map(circle); axes[1].plot(ellipse[:, 0], ellipse[:, 1], color=PALETTE["blue"], lw=2.2, label="circle image")
axes[1].set_title("After: affine map u -> M u + c", color=PALETTE["ink"], weight="bold"); axes[1].set_xlim(-1.0, 3.0); axes[1].set_ylim(-1.55, 1.95)
A = np.array([-0.75, -0.55]); B = np.array([0.85, 0.65]); t = 0.35; C = (1 - t) * A + t * B; mid = 0.5 * (A + B)
for ax, mapper in [(axes[0], lambda x: x), (axes[1], affine_map)]:
    img = mapper(np.vstack([A, C, mid, B])); ax.plot([img[0, 0], img[-1, 0]], [img[0, 1], img[-1, 1]], color=PALETTE["red"], lw=2.3)
    for point, label, color in zip(img, ["A", "ratio point", "midpoint", "B"], [PALETTE["red"], PALETTE["orange"], PALETTE["green"], PALETTE["red"]]):
        ax.scatter(point[0], point[1], s=52, color=color, zorder=5); ax.text(point[0] + 0.04, point[1] + 0.04, label, fontsize=8.8, color=PALETTE["ink"])
for ax in axes: ax.legend(loc="upper left", fontsize=8, frameon=True)
fig.suptitle("Affine Geometry Keeps Linear Relations, Not Euclidean Measurements", fontsize=15, weight="bold", color=PALETTE["ink"])
affine_fig = save_figure(fig, "affine-vector-grid-invariants.png")
A_img, B_img, C_img, mid_img = affine_map(np.vstack([A, B, C, mid])); ratio_img = np.linalg.norm(C_img - A_img) / np.linalg.norm(B_img - A_img)
M_sym = sp.Matrix([[sp.Rational(27, 20), sp.Rational(29, 50)], [sp.Rational(-7, 25), sp.Rational(23, 25)]])
chapter_checks["affine_vector_transformation"] = {"determinant_area_scale_numeric": float(np.linalg.det(M_aff)), "determinant_area_scale_exact": str(sp.factor(M_sym.det())), "midpoint_residual": float(np.linalg.norm(mid_img - 0.5 * (A_img + B_img))), "ratio_before": float(t), "ratio_after": float(ratio_img), "ratio_residual": float(abs(ratio_img - t)), "unit_x_length_after": float(np.linalg.norm(M_aff @ np.array([1.0, 0.0]))), "unit_y_length_after": float(np.linalg.norm(M_aff @ np.array([0.0, 1.0])))}
display_artifact(affine_fig, width=900)
chapter_checks["affine_vector_transformation"]


## 4. Projective-Line Maps from Ordinary Matrices

A point of `RP^1` is a line through the origin in `R^2`. Label the non-horizontal line through `(s, 1)` by `s`; label the horizontal line by infinity. A matrix

`[[a, b], [c, d]]`

sends the direction `(s, 1)` to `(a s + b, c s + d)`, so it sends the label `s` to `(a s + b)/(c s + d)` whenever the denominator is nonzero. This is the linear fractional map from the projective chapters, now recast as a transformation-group example.

The important warning is that the matrix is not the geometry. Multiplying the matrix by a nonzero scalar changes the linear map of `R^2`, but it does not change the projective map on lines through the origin. The invariant that survives the projective-line group is cross-ratio.


In [ ]:
def mobius_exact(s):
    return (2 * s - 1) / (s + 3)

def cross_ratio_exact(values):
    a, b, c, d = values
    return sp.factor(((a - c) * (b - d)) / ((a - d) * (b - c)))

sample_s = [sp.Rational(-2, 1), sp.Rational(-1, 2), sp.Rational(1, 1), sp.Rational(4, 1)]
sample_img = [sp.simplify(mobius_exact(s)) for s in sample_s]
cr_before = cross_ratio_exact(sample_s); cr_after = cross_ratio_exact(sample_img)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6), facecolor=PALETTE["paper"])
colors = [PALETTE["blue"], PALETTE["orange"], PALETTE["green"], PALETTE["red"]]

def draw_rp1_panel(ax, values, title):
    equal_2d(ax); ax.axhline(0, color=PALETTE["ink"], lw=1.0); ax.axvline(0, color=PALETTE["ink"], lw=1.0); ax.axhline(1, color=PALETTE["gray"], lw=1.0, ls="--")
    ax.text(-3.2, 1.06, "label line y = 1", fontsize=8.5, color=PALETTE["gray"])
    for value, color in zip(values, colors):
        sv = float(value); direction = np.array([sv, 1.0]); direction = direction / np.linalg.norm(direction)
        line = np.vstack([-3.2 * direction, 3.2 * direction]); ax.plot(line[:, 0], line[:, 1], color=color, lw=2.2)
        ax.scatter([sv], [1.0], color=color, s=48, zorder=5); ax.text(sv + 0.07, 1.08, f"{sp.sstr(value)}", color=color, fontsize=9, weight="bold")
    ax.set_xlim(-3.4, 3.4); ax.set_ylim(-1.9, 2.25); ax.set_title(title, color=PALETTE["ink"], weight="bold")

draw_rp1_panel(axes[0], sample_s, "Before: labels s on RP^1")
draw_rp1_panel(axes[1], sample_img, "After: f(s) = (2s - 1)/(s + 3)")
fig.suptitle("A Projective-Line Map Moves Lines, Not Chosen Points", fontsize=15, weight="bold", color=PALETTE["ink"])
fig.text(0.17, 0.02, f"cross-ratio before = {sp.sstr(cr_before)}", color=PALETTE["ink"], fontsize=11)
fig.text(0.58, 0.02, f"cross-ratio after = {sp.sstr(cr_after)}", color=PALETTE["ink"], fontsize=11)
projective_fig = save_figure(fig, "projective-line-mobius-cross-ratio.png")
M_proj = np.array([[2.0, -1.0], [1.0, 3.0]]); k = -2.75; scalar_residuals = []; line_label_residuals = []
for s in [float(v) for v in sample_s]:
    v = np.array([s, 1.0]); label_M = (M_proj @ v)[0] / (M_proj @ v)[1]; label_kM = ((k * M_proj) @ v)[0] / ((k * M_proj) @ v)[1]
    scalar_residuals.append(abs(label_M - label_kM))
    for scale in [0.4, 1.0, 2.3]:
        label_scaled_point = (M_proj @ (scale * v))[0] / (M_proj @ (scale * v))[1]; line_label_residuals.append(abs(label_scaled_point - label_M))
x1 = np.linspace(-8.0, -3.08, 250); x2 = np.linspace(-2.92, 8.0, 350)
y1 = (2 * x1 - 1) / (x1 + 3); y2 = (2 * x2 - 1) / (x2 + 3)
fig_plotly = go.Figure()
fig_plotly.add_trace(go.Scatter(x=x1, y=y1, mode="lines", name="f(s), left of pole", line={"color": PALETTE["purple"]}))
fig_plotly.add_trace(go.Scatter(x=x2, y=y2, mode="lines", name="f(s), right of pole", line={"color": PALETTE["purple"]}))
fig_plotly.add_trace(go.Scatter(x=[float(s) for s in sample_s], y=[float(v) for v in sample_img], mode="markers+text", text=[sp.sstr(s) for s in sample_s], textposition="top center", name="sample labels", marker={"size": 9, "color": "#c74343"}))
fig_plotly.add_vline(x=-3, line_dash="dash", line_color="#78818c", annotation_text="denominator 0")
fig_plotly.update_layout(title="Projective-line map f(s) = (2s - 1)/(s + 3)", xaxis_title="label s", yaxis_title="label f(s)", template="plotly_white", height=460)
projective_html = artifact_path(ARTIFACT_ROOT, "html", "projective-line-map-explorer.html")
fig_plotly.write_html(str(projective_html), include_plotlyjs=True, full_html=True); html_paths.append(projective_html)
chapter_checks["projective_line_map"] = {"cross_ratio_before_exact": str(cr_before), "cross_ratio_after_exact": str(cr_after), "cross_ratio_exact_match": bool(sp.simplify(cr_before - cr_after) == 0), "scalar_matrix_projective_label_max_error": float(max(scalar_residuals)), "same_line_scaled_point_label_max_error": float(max(line_label_residuals)), "matrix_determinant": float(np.linalg.det(M_proj))}
display_artifact(projective_fig, width=900)
display_artifact(projective_html, width="100%", height=430)
chapter_checks["projective_line_map"]


## 5. Spherical Geometry and Rotations

On the sphere, a line is a great circle: the intersection of `S^2` with a plane through the origin. That definition makes spherical geometry more inspectable in three dimensions than in a flat drawing. A reflection in such a plane fixes the corresponding great circle. The product of two plane reflections is a rotation around the line where the planes meet.

The static image records the construction, and the HTML artifact lets you rotate the view. Watch the red geodesic arc before and after the product rotation: its spherical length is computed by `arccos(P dot Q)`, not by the straight chord length.


In [ ]:
def normalize(v):
    arr = np.asarray(v, dtype=float); nrm = np.linalg.norm(arr)
    if nrm == 0: raise ValueError("zero vector")
    return arr / nrm

def plane_reflection(normal):
    n = normalize(normal); return np.eye(3) - 2.0 * np.outer(n, n)

def plane_basis(normal):
    n = normalize(normal); trial = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.85 else np.array([0.0, 1.0, 0.0])
    u = normalize(np.cross(n, trial)); v = normalize(np.cross(n, u)); return u, v

def great_circle(normal, samples=240):
    u, v = plane_basis(normal); t = np.linspace(0, 2 * math.pi, samples)
    return np.outer(np.cos(t), u) + np.outer(np.sin(t), v)

def slerp(p, q, samples=120):
    p = normalize(p); q = normalize(q); omega = math.acos(float(np.clip(np.dot(p, q), -1.0, 1.0)))
    t = np.linspace(0, 1, samples)
    if omega < 1e-12: return np.tile(p, (samples, 1))
    return (np.sin((1 - t)[:, None] * omega) * p + np.sin(t[:, None] * omega) * q) / math.sin(omega)

n1 = normalize([0.3, -0.7, 0.64]); n2 = normalize([-0.55, -0.2, 0.81])
H1 = plane_reflection(n1); H2 = plane_reflection(n2); sphere_rotation = H2 @ H1; axis = normalize(np.cross(n1, n2))
P = normalize([0.72, -0.18, 0.67]); Q = normalize([-0.12, 0.84, 0.52])
P_rot = sphere_rotation @ P; Q_rot = sphere_rotation @ Q; arc_before = slerp(P, Q); arc_after = slerp(P_rot, Q_rot)
fig = plt.figure(figsize=(10.5, 8), facecolor=PALETTE["paper"]); ax = fig.add_subplot(111, projection="3d", facecolor=PALETTE["paper"])
u = np.linspace(0, 2 * math.pi, 80); v = np.linspace(0, math.pi, 40)
xs = np.outer(np.cos(u), np.sin(v)); ys = np.outer(np.sin(u), np.sin(v)); zs = np.outer(np.ones_like(u), np.cos(v))
ax.plot_surface(xs, ys, zs, color="#dbeaf7", alpha=0.24, linewidth=0, shade=False)
for normal, color, label in [(n1, PALETTE["blue"], "reflection plane 1"), (n2, PALETTE["orange"], "reflection plane 2")]:
    gc = great_circle(normal); ax.plot(gc[:, 0], gc[:, 1], gc[:, 2], color=color, lw=2.3, label=label)
ax.plot(arc_before[:, 0], arc_before[:, 1], arc_before[:, 2], color=PALETTE["red"], lw=3.2, label="great-circle arc P-Q")
ax.plot(arc_after[:, 0], arc_after[:, 1], arc_after[:, 2], color=PALETTE["green"], lw=3.2, label="rotated arc")
for point, label, color in [(P, "P", PALETTE["red"]), (Q, "Q", PALETTE["red"]), (P_rot, "P'", PALETTE["green"]), (Q_rot, "Q'", PALETTE["green"] )]:
    ax.scatter([point[0]], [point[1]], [point[2]], s=58, color=color); ax.text(point[0] * 1.08, point[1] * 1.08, point[2] * 1.08, label, color=PALETTE["ink"], fontsize=10)
axis_line = np.vstack([-1.25 * axis, 1.25 * axis]); ax.plot(axis_line[:, 0], axis_line[:, 1], axis_line[:, 2], color=PALETTE["purple"], lw=3.0, label="rotation axis")
ax.set_title("Spherical Isometries from Great-Circle Reflections", fontsize=15, weight="bold", color=PALETTE["ink"]); ax.set_box_aspect((1, 1, 1))
ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15); ax.set_zlim(-1.15, 1.15); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.view_init(elev=22, azim=38); ax.legend(loc="upper left", fontsize=9)
sphere_fig = save_figure(fig, "spherical-great-circle-rotations.png")
fig3 = go.Figure()
fig3.add_trace(go.Surface(x=xs, y=ys, z=zs, opacity=0.22, showscale=False, colorscale=[[0, "#dbeaf7"], [1, "#dbeaf7"]], name="S^2"))
for normal, color, label in [(n1, PALETTE["blue"], "great circle 1"), (n2, PALETTE["orange"], "great circle 2")]:
    gc = great_circle(normal); fig3.add_trace(go.Scatter3d(x=gc[:,0], y=gc[:,1], z=gc[:,2], mode="lines", line={"color": color, "width": 6}, name=label))
fig3.add_trace(go.Scatter3d(x=arc_before[:,0], y=arc_before[:,1], z=arc_before[:,2], mode="lines", line={"color": PALETTE["red"], "width": 7}, name="arc before"))
fig3.add_trace(go.Scatter3d(x=arc_after[:,0], y=arc_after[:,1], z=arc_after[:,2], mode="lines", line={"color": PALETTE["green"], "width": 7}, name="arc after"))
fig3.add_trace(go.Scatter3d(x=axis_line[:,0], y=axis_line[:,1], z=axis_line[:,2], mode="lines", line={"color": PALETTE["purple"], "width": 8}, name="rotation axis"))
for point, label, color in [(P, "P", PALETTE["red"]), (Q, "Q", PALETTE["red"]), (P_rot, "P prime", PALETTE["green"]), (Q_rot, "Q prime", PALETTE["green"] )]:
    fig3.add_trace(go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode="markers+text", text=[label], textposition="top center", marker={"size": 5, "color": color}, name=label))
fig3.update_layout(title="Interactive spherical reflection product", template="plotly_white", height=620, scene={"aspectmode": "cube"})
sphere_html = artifact_path(ARTIFACT_ROOT, "html", "spherical-rotation-explorer.html")
fig3.write_html(str(sphere_html), include_plotlyjs=True, full_html=True); html_paths.append(sphere_html)
spherical_distance_before = math.acos(float(np.clip(np.dot(P, Q), -1.0, 1.0))); spherical_distance_after = math.acos(float(np.clip(np.dot(P_rot, Q_rot), -1.0, 1.0)))
chapter_checks["spherical_geometry"] = {"rotation_orthogonality_residual": float(np.linalg.norm(sphere_rotation.T @ sphere_rotation - np.eye(3))), "rotation_determinant": float(np.linalg.det(sphere_rotation)), "point_norm_max_error": float(max(abs(np.linalg.norm(P_rot) - 1), abs(np.linalg.norm(Q_rot) - 1))), "great_circle_distance_before": float(spherical_distance_before), "great_circle_distance_after": float(spherical_distance_after), "great_circle_distance_residual": float(abs(spherical_distance_before - spherical_distance_after))}
display_artifact(sphere_fig, width=820)
display_artifact(sphere_html, width="100%", height=520)
chapter_checks["spherical_geometry"]


## 6. Quaternion Rotations and the Double Cover

A unit quaternion `q = cos(theta/2) + u sin(theta/2)` rotates imaginary quaternions by the sandwich action `p -> q p q^{-1}`. The half-angle is not cosmetic: the left and right multiplication combine to create a full turn by `theta` in three-space.

The same rotation is represented by both `q` and `-q`. This is the double cover from `S^3` to the rotation group. It is the computational reason `RP^3`, whose points identify antipodal points of `S^3`, is the right geometric object for rotations of the sphere.


In [ ]:
def rodrigues(axis, theta):
    a = normalize(axis); K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]], dtype=float)
    return np.eye(3) + math.sin(theta) * K + (1 - math.cos(theta)) * (K @ K)
axis_q = normalize([0.45, -0.35, 0.82]); theta_q = math.radians(112)
q = q_from_axis_angle(axis_q, theta_q); q_neg = -q; basis = np.eye(3)
rotated_basis_q = np.array([q_rotate(q, v) for v in basis]); rotated_basis_neg = np.array([q_rotate(q_neg, v) for v in basis])
R_rodrigues = rodrigues(axis_q, theta_q); rotated_basis_matrix = (R_rodrigues @ basis.T).T
fig = plt.figure(figsize=(13, 5.8), facecolor=PALETTE["paper"])
ax1 = fig.add_subplot(1, 2, 1, projection="3d", facecolor=PALETTE["paper"])
colors = [PALETTE["red"], PALETTE["green"], PALETTE["blue"]]; labels = ["e1", "e2", "e3"]
for v, color, label in zip(basis, colors, labels):
    ax1.quiver(0, 0, 0, v[0], v[1], v[2], color=color, alpha=0.35, linewidth=2, arrow_length_ratio=0.08); ax1.text(*(1.08 * v), label, color=color, fontsize=9)
for v, color, label in zip(rotated_basis_q, colors, ["q e1 q^-1", "q e2 q^-1", "q e3 q^-1"]):
    ax1.quiver(0, 0, 0, v[0], v[1], v[2], color=color, linewidth=3, arrow_length_ratio=0.08); ax1.text(*(1.15 * v), label, color=PALETTE["ink"], fontsize=8)
ax1.quiver(0, 0, 0, axis_q[0], axis_q[1], axis_q[2], color=PALETTE["purple"], linewidth=4, arrow_length_ratio=0.08); ax1.text(*(1.18 * axis_q), "axis u", color=PALETTE["purple"], fontsize=10, weight="bold")
ax1.set_title("Quaternion sandwich action rotates a frame", color=PALETTE["ink"], weight="bold"); ax1.set_box_aspect((1, 1, 1)); ax1.set_xlim(-1.05, 1.05); ax1.set_ylim(-1.05, 1.05); ax1.set_zlim(-1.05, 1.05); ax1.view_init(elev=20, azim=40)
ax2 = fig.add_subplot(1, 2, 2, facecolor=PALETTE["paper"]); equal_2d(ax2)
phis = np.linspace(0, 2 * math.pi, 300); ax2.plot(np.cos(phis), np.sin(phis), color=PALETTE["gray"], lw=1.5); ax2.axhline(0, color=PALETTE["gray"], lw=0.9); ax2.axvline(0, color=PALETTE["gray"], lw=0.9)
ax2.scatter([q[0], q_neg[0]], [np.linalg.norm(q[1:]), -np.linalg.norm(q[1:])], s=95, color=[PALETTE["green"], PALETTE["red"]], zorder=5)
ax2.text(q[0] + 0.05, np.linalg.norm(q[1:]) + 0.05, "q", color=PALETTE["green"], weight="bold"); ax2.text(q_neg[0] - 0.2, -np.linalg.norm(q[1:]) - 0.12, "-q", color=PALETTE["red"], weight="bold")
ax2.plot([q[0], q_neg[0]], [np.linalg.norm(q[1:]), -np.linalg.norm(q[1:])], color=PALETTE["purple"], ls="--", lw=1.8)
ax2.set_xlabel("scalar part cos(theta/2)"); ax2.set_ylabel("signed vector-axis coefficient"); ax2.set_title("Antipodal unit quaternions give one rotation", color=PALETTE["ink"], weight="bold"); ax2.set_xlim(-1.2, 1.2); ax2.set_ylim(-1.2, 1.2)
fig.suptitle("Quaternion Rotations: Half-Angle Coordinates and Double Cover", fontsize=15, weight="bold", color=PALETTE["ink"])
quat_fig = save_figure(fig, "quaternion-double-cover-axis-angle.png")
chapter_checks["quaternion_rotation"] = {"q_norm": float(np.linalg.norm(q)), "negative_q_same_rotation_residual": float(np.linalg.norm(rotated_basis_q - rotated_basis_neg)), "rodrigues_matrix_residual": float(np.linalg.norm(rotated_basis_q - rotated_basis_matrix)), "rotated_basis_orthonormal_residual": float(np.linalg.norm(rotated_basis_q @ rotated_basis_q.T - np.eye(3)))}
display_artifact(quat_fig, width=900)
chapter_checks["quaternion_rotation"]


## 7. Applied Rotation Lab: Order Matters in a Moving Frame

A practical rotation task rarely consists of one clean turn about one clean axis. A camera, spacecraft sensor, robot wrist, or game object often receives a sequence of commands. Quaternions are useful here because composition is just quaternion multiplication, while the resulting frame remains orthonormal when the inputs are unit quaternions.

The lab compares two command orders. Apply a yaw about the world `z` axis and a pitch about the world `x` axis. The same two angles give different final body frames when the order changes. This is the applied version of the chapter's warning that rotations of `S^2` usually do not commute.


In [ ]:
yaw = math.radians(70); pitch = math.radians(55)
q_yaw = q_from_axis_angle([0, 0, 1], yaw); q_pitch = q_from_axis_angle([1, 0, 0], pitch)
q_yaw_then_pitch = q_multiply(q_pitch, q_yaw); q_pitch_then_yaw = q_multiply(q_yaw, q_pitch)
frame0 = np.eye(3); frame_yaw_pitch = np.array([q_rotate(q_yaw_then_pitch, v) for v in frame0]); frame_pitch_yaw = np.array([q_rotate(q_pitch_then_yaw, v) for v in frame0])
def add_frame(fig, frame, name, colors, dash=None):
    for vec, color, axis_name in zip(frame, colors, ["x", "y", "z"]):
        fig.add_trace(go.Scatter3d(x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]], mode="lines+markers+text", text=["", f"{name} {axis_name}"], textposition="top center", line={"color": color, "width": 7, "dash": dash or "solid"}, marker={"size": 3, "color": color}, name=f"{name} {axis_name}"))
lab_fig = go.Figure(); add_frame(lab_fig, frame0, "start", ["#9aa4af", "#9aa4af", "#9aa4af"], dash="dot")
add_frame(lab_fig, frame_yaw_pitch, "yaw then pitch", [PALETTE["red"], PALETTE["green"], PALETTE["blue"]])
add_frame(lab_fig, frame_pitch_yaw, "pitch then yaw", ["#a23b72", "#2ca58d", "#2d6cdf"], dash="dash")
lab_fig.update_layout(title="Applied quaternion rotation lab: same commands, different order", template="plotly_white", height=620, scene={"aspectmode": "cube", "xaxis": {"range": [-1.1, 1.1]}, "yaxis": {"range": [-1.1, 1.1]}, "zaxis": {"range": [-1.1, 1.1]}}, legend={"itemsizing": "constant"})
lab_html = artifact_path(ARTIFACT_ROOT, "html", "applied-quaternion-rotation-lab.html")
lab_fig.write_html(str(lab_html), include_plotlyjs=True, full_html=True); html_paths.append(lab_html)
R_yaw_pitch = np.column_stack(frame_yaw_pitch); R_pitch_yaw = np.column_stack(frame_pitch_yaw)
chapter_checks["applied_rotation_lab"] = {"yaw_degrees": float(math.degrees(yaw)), "pitch_degrees": float(math.degrees(pitch)), "yaw_pitch_orthonormal_residual": float(np.linalg.norm(R_yaw_pitch.T @ R_yaw_pitch - np.eye(3))), "pitch_yaw_orthonormal_residual": float(np.linalg.norm(R_pitch_yaw.T @ R_pitch_yaw - np.eye(3))), "yaw_pitch_determinant": float(np.linalg.det(R_yaw_pitch)), "pitch_yaw_determinant": float(np.linalg.det(R_pitch_yaw)), "order_difference_norm": float(np.linalg.norm(frame_yaw_pitch - frame_pitch_yaw))}
display_artifact(lab_html, width="100%", height=520)
chapter_checks["applied_rotation_lab"]


## 8. A Finite Rotation Group: Tetrahedron to 24 Unit Quaternions

The tetrahedron has 12 rotational symmetries: the identity, three half-turns about axes through opposite edge centers, and eight one-third turns about vertex-to-opposite-face axes. The quaternion representation lifts those 12 rotations to 24 unit quaternions because every rotation is represented by an antipodal pair `+q` and `-q`.

The right panel plots a fixed linear projection of those 24 quaternions from `R^4` into `R^3`. This is not a literal view of four-dimensional space. It is a faithful bookkeeping diagram for the counts, types, antipodal pairing, and unit-norm condition that make the 24-cell appear in the chapter.


In [ ]:
tetra_vertices = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]], dtype=float) / math.sqrt(3)
tetra_faces = np.array([[0, 1, 2], [0, 3, 1], [0, 2, 3], [1, 3, 2]])
tetra_mesh = trimesh.Trimesh(vertices=tetra_vertices, faces=tetra_faces, process=False)
axis_quats, q_labels = [], []
axis_quats.extend([[1, 0, 0, 0], [-1, 0, 0, 0]]); q_labels.extend(["identity", "identity"])
for idx in range(1, 4):
    for sign in [1, -1]:
        qv = [0, 0, 0, 0]; qv[idx] = sign; axis_quats.append(qv); q_labels.append("half-turn")
for signs in np.array(np.meshgrid([1, -1], [1, -1], [1, -1], [1, -1])).T.reshape(-1, 4):
    axis_quats.append((0.5 * signs).tolist()); q_labels.append("third-turn")
Q24 = np.array(axis_quats, dtype=float)
projection = np.array([[0.42, -0.20, 0.18], [1.00, 0.18, -0.12], [-0.15, 0.95, 0.24], [0.12, -0.28, 0.98]], dtype=float)
Q3 = Q24 @ projection; label_colors = {"identity": PALETTE["ink"], "half-turn": PALETTE["orange"], "third-turn": PALETTE["purple"]}
fig = plt.figure(figsize=(13, 6.2), facecolor=PALETTE["paper"])
ax1 = fig.add_subplot(1, 2, 1, projection="3d", facecolor=PALETTE["paper"])
poly = Poly3DCollection([tetra_vertices[face] for face in tetra_faces], alpha=0.18, facecolor=PALETTE["blue"], edgecolor=PALETTE["ink"], linewidth=1.2); ax1.add_collection3d(poly)
for edge in tetra_mesh.edges_unique:
    pts = tetra_vertices[edge]; ax1.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=PALETTE["ink"], lw=1.8)
ax1.scatter(tetra_vertices[:, 0], tetra_vertices[:, 1], tetra_vertices[:, 2], s=58, color=PALETTE["blue"])
for i, vtx in enumerate(tetra_vertices): ax1.text(*(1.08 * vtx), f"v{i+1}", color=PALETTE["ink"], fontsize=9)
for direction, color, label in [([1,0,0], PALETTE["orange"], "1/2 axes"), ([0,1,0], PALETTE["orange"], ""), ([0,0,1], PALETTE["orange"], "")]:
    d = np.array(direction, dtype=float); line = np.vstack([-1.25 * d, 1.25 * d]); ax1.plot(line[:,0], line[:,1], line[:,2], color=color, lw=2.6, label=label if label else None)
for vtx in tetra_vertices:
    line = np.vstack([-1.15 * vtx, 1.15 * vtx]); ax1.plot(line[:,0], line[:,1], line[:,2], color=PALETTE["green"], lw=1.7, alpha=0.85)
ax1.plot([], [], [], color=PALETTE["green"], lw=2, label="1/3 axes")
ax1.set_title("Tetrahedron rotation axes", color=PALETTE["ink"], weight="bold"); ax1.set_box_aspect((1, 1, 1)); ax1.set_xlim(-1.25, 1.25); ax1.set_ylim(-1.25, 1.25); ax1.set_zlim(-1.25, 1.25); ax1.view_init(elev=22, azim=36); ax1.legend(loc="upper left", fontsize=9)
ax2 = fig.add_subplot(1, 2, 2, projection="3d", facecolor=PALETTE["paper"])
for kind in ["identity", "half-turn", "third-turn"]:
    mask = np.array(q_labels) == kind; ax2.scatter(Q3[mask, 0], Q3[mask, 1], Q3[mask, 2], s=70, color=label_colors[kind], label=kind, depthshade=False)
for point in Q3:
    opposite = Q3[np.argmin(np.linalg.norm(Q3 + point, axis=1))]
    ax2.plot([point[0], opposite[0]], [point[1], opposite[1]], [point[2], opposite[2]], color=PALETTE["gray"], alpha=0.12, lw=0.8)
ax2.set_title("24 unit quaternions: a projected 24-cell vertex set", color=PALETTE["ink"], weight="bold"); ax2.set_box_aspect((1, 1, 1)); ax2.view_init(elev=20, azim=-52); ax2.legend(loc="upper left", fontsize=9)
fig.suptitle("Finite Rotation Group of the Tetrahedron", fontsize=15, weight="bold", color=PALETTE["ink"])
tetra_fig = save_figure(fig, "tetrahedral-rotation-quaternion-cloud.png")
def canonical_pair(qv):
    qv = np.asarray(qv, dtype=float)
    for value in qv:
        if abs(value) > 1e-12:
            if value < 0: qv = -qv
            break
    return tuple(np.round(qv, 12))
pairs = {canonical_pair(qv) for qv in Q24}
chapter_checks["finite_tetrahedral_rotation_group"] = {"unit_quaternion_count": int(len(Q24)), "antipodal_pair_count": int(len(pairs)), "max_unit_norm_error": float(np.max(np.abs(np.linalg.norm(Q24, axis=1) - 1.0))), "identity_quaternion_count": int(q_labels.count("identity")), "half_turn_quaternion_count": int(q_labels.count("half-turn")), "third_turn_quaternion_count": int(q_labels.count("third-turn")), "tetrahedron_mesh_euler_number": int(tetra_mesh.euler_number), "tetrahedron_edge_count": int(len(tetra_mesh.edges_unique)), "tetrahedron_volume": float(tetra_mesh.volume)}
display_artifact(tetra_fig, width=900)
chapter_checks["finite_tetrahedral_rotation_group"]


## 9. `S^3`, `RP^3`, and the Rotation Group

The unit quaternions form the 3-sphere `S^3` inside `R^4`. Multiplication of unit quaternions stays on `S^3`, so `S^3` is itself a group. Rotations of three-space are not exactly `S^3`, because antipodal unit quaternions act the same way on imaginary quaternions. Once antipodal points are identified, the quotient is `RP^3`, and that space models the rotation group `SO(3)`.

The fixed-axis slice below shows the essential topology without pretending to draw all of `S^3`. A rotation angle of `2*pi` lands at `-1` in `S^3`, not back at `1`. In `RP^3`, those antipodal points are identified, so the physical rotation is back at the identity.


In [ ]:
axis_fixed = normalize([0.0, 0.0, 1.0]); thetas = np.linspace(0, 4 * math.pi, 500); slice_points = np.c_[np.cos(thetas / 2), np.sin(thetas / 2)]
fig, ax = plt.subplots(figsize=(8.4, 7.2), facecolor=PALETTE["paper"]); equal_2d(ax)
ax.plot(slice_points[:, 0], slice_points[:, 1], color=PALETTE["purple"], lw=2.4)
unit_circle = np.c_[np.cos(np.linspace(0, 2 * math.pi, 400)), np.sin(np.linspace(0, 2 * math.pi, 400))]; ax.plot(unit_circle[:, 0], unit_circle[:, 1], color=PALETTE["gray"], lw=1.0, alpha=0.8)
for theta, label, color in [(0, "0 turn: q=1", PALETTE["green"]), (math.pi, "pi turn", PALETTE["blue"]), (2 * math.pi, "2pi turn: q=-1", PALETTE["red"]), (4 * math.pi, "4pi turn: q=1", PALETTE["green"])]:
    point = np.array([math.cos(theta / 2), math.sin(theta / 2)]); ax.scatter(point[0], point[1], s=88, color=color, zorder=5); ax.text(point[0] + 0.06, point[1] + 0.06, label, color=PALETTE["ink"], fontsize=9, weight="bold")
ax.plot([1, -1], [0, 0], color=PALETTE["red"], lw=2, ls="--"); ax.text(-0.44, -0.14, "antipodal points identified in RP^3", color=PALETTE["red"], fontsize=10, ha="center")
ax.set_xlabel("scalar part"); ax.set_ylabel("fixed-axis coefficient"); ax.set_title("A Fixed-Axis Slice of S^3 and the RP^3 Identification", fontsize=15, weight="bold", color=PALETTE["ink"]); ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.25)
s3_fig = save_figure(fig, "s3-rp3-antipodal-model.png")
q0 = q_from_axis_angle(axis_fixed, 0.0); q2 = q_from_axis_angle(axis_fixed, 2 * math.pi); q4 = q_from_axis_angle(axis_fixed, 4 * math.pi); vtest = np.array([1.0, 0.35, -0.2])
product_norm_sample = np.linalg.norm(q_multiply(q_from_axis_angle([1,0,0], 0.73), q_from_axis_angle([0,1,0], -1.12)))
chapter_checks["s3_rp3_rotation_group"] = {"s3_distance_q0_to_q2pi": float(np.linalg.norm(q0 - q2)), "s3_distance_q0_to_q4pi": float(np.linalg.norm(q0 - q4)), "rotation_residual_0_vs_2pi": float(np.linalg.norm(q_rotate(q0, vtest) - q_rotate(q2, vtest))), "rotation_residual_0_vs_4pi": float(np.linalg.norm(q_rotate(q0, vtest) - q_rotate(q4, vtest))), "sample_product_unit_norm": float(product_norm_sample)}
display_artifact(s3_fig, width=760)
chapter_checks["s3_rp3_rotation_group"]


## Final Sanity Checks

The final cell collects the chapter's invariant data and asserts the claims that the visuals rely on. These are not exhaustive proofs. They are executable guardrails: a broken quaternion sign convention, stale artifact path, blank figure, or failed cross-ratio calculation should stop the notebook.


In [ ]:
transform_checks_path = save_json(chapter_checks, ARTIFACT_ROOT, "checks", "transformation-invariants.json"); check_paths.append(transform_checks_path)
quaternion_checks = {key: chapter_checks[key] for key in ["quaternion_rotation", "applied_rotation_lab", "finite_tetrahedral_rotation_group", "s3_rp3_rotation_group"]}
quaternion_checks_path = save_json(quaternion_checks, ARTIFACT_ROOT, "checks", "quaternion-rotation-checks.json"); check_paths.append(quaternion_checks_path)
manifest_path = ARTIFACT_ROOT / "checks" / "artifact-manifest.json"
manifest = {"chapter_key": CHAPTER_KEY, "source_span": "printed pages 143-173; course-mapped PDF pages 153-183", "figures": [rel(path) for path in figure_paths], "html": [rel(path) for path in html_paths], "tables": [rel(path) for path in table_paths], "checks": [rel(path) for path in [*check_paths, manifest_path]], "storyboard_items": ["group invariants map", "plane isometry reflection composition", "affine/vector grid invariants", "projective-line Mobius cross-ratio map", "spherical great-circle reflection rotations", "quaternion rotation double cover", "applied quaternion rotation lab", "finite tetrahedral rotation group and 24-cell vertices", "S^3 and RP^3 antipodal model"]}
manifest_path = save_json(manifest, ARTIFACT_ROOT, "checks", "artifact-manifest.json"); check_paths.append(manifest_path)
all_paths = figure_paths + html_paths + table_paths + check_paths
assert_artifacts(all_paths, min_size=128)
stats = [image_stats(path) for path in figure_paths]
assert all(item["width"] >= 300 and item["height"] >= 240 for item in stats)
assert all(item["pixel_std"] > 2.0 for item in stats)
assert chapter_checks["plane_isometry_composition"]["distance_max_error"] < 1e-12
assert abs(chapter_checks["plane_isometry_composition"]["det_reflection_alpha"] + 1.0) < 1e-12
assert abs(chapter_checks["plane_isometry_composition"]["det_two_reflections"] - 1.0) < 1e-12
assert chapter_checks["affine_vector_transformation"]["midpoint_residual"] < 1e-12
assert chapter_checks["affine_vector_transformation"]["ratio_residual"] < 1e-12
assert chapter_checks["projective_line_map"]["cross_ratio_exact_match"]
assert chapter_checks["projective_line_map"]["scalar_matrix_projective_label_max_error"] < 1e-12
assert chapter_checks["spherical_geometry"]["great_circle_distance_residual"] < 1e-12
assert chapter_checks["spherical_geometry"]["rotation_orthogonality_residual"] < 1e-12
assert chapter_checks["quaternion_rotation"]["negative_q_same_rotation_residual"] < 1e-12
assert chapter_checks["quaternion_rotation"]["rodrigues_matrix_residual"] < 1e-12
assert chapter_checks["applied_rotation_lab"]["order_difference_norm"] > 0.25
assert chapter_checks["finite_tetrahedral_rotation_group"]["unit_quaternion_count"] == 24
assert chapter_checks["finite_tetrahedral_rotation_group"]["antipodal_pair_count"] == 12
assert chapter_checks["finite_tetrahedral_rotation_group"]["max_unit_norm_error"] < 1e-12
assert chapter_checks["s3_rp3_rotation_group"]["rotation_residual_0_vs_2pi"] < 1e-12
assert abs(chapter_checks["s3_rp3_rotation_group"]["sample_product_unit_norm"] - 1.0) < 1e-12
print(f"Generated {len(figure_paths)} figures, {len(html_paths)} HTML artifacts, {len(table_paths)} tables, and {len(check_paths)} check files.")
print("All Chapter 7 transformation sanity checks passed.")
manifest


## Takeaways

- A geometry can be read from its transformation group by asking which properties every allowed transformation preserves.
- Plane isometries protect distance; the orientation-preserving subgroup protects handedness or cyclic order that all isometries do not.
- Linear maps preserve vector operations and the origin. Affine maps remove the special role of the origin while preserving parallelism, midpoints, centroids, and ratios along a line.
- Projective-line transformations are linear fractional maps because ordinary matrices move lines through the origin. Matrix scaling is invisible projectively, and cross-ratio is the surviving numeric invariant.
- Spherical lines are great circles. Reflections in great circles generate spherical isometries, and products of two reflections are rotations.
- Unit quaternions represent rotations by `p -> q p q^{-1}`. The signs `q` and `-q` are distinct points of `S^3` but the same rotation of three-space.
- The tetrahedral rotation group shows the bridge from finite symmetry to four-dimensional geometry: 12 rotations lift to 24 unit quaternions, a vertex set of the 24-cell.
- The full rotation group of the sphere is modeled by `RP^3`, while `S^3` is the double cover that makes quaternion multiplication continuous and easy to compute.
